# Project 5 -- Boqiang Zhang

**TA Help:** None

**Collaboration:** None

**Internet Resources:** None

**ChatGPT, Gemini, Claude, etc:** Gemini, link: TBD

- In Q1, I asked Gemini about the step 'logits = self(x)' and what logits mean
- In Q2, I asked Gemini about what .dropna does in pandas


**OVERALL MESSAGE:** Any time that you used anything except your brain to solve the questions in these projects, you need to disclose such resources at the start of the project, with details about your usage of the tools.

**YOUR OWN WORK:** Even when you utilize other resources, do NOT just copy and paste.  Write all explanations in your own words, using several sentences in English, which are understandable and which you wrote (and did not just copy and paste).

## Question 1

### 1.1 Two-three sentences explaining the general flow and benefits of this setup.

- We use instead a class to define the training and verification process. This makes it more modular and easier to maintain. It can also benefit from pre-defined classes (e.g. LightningModule) to use optimized functions and attributes.

### 1.2 Run `!cat {your_path_to_project}/intro_mlops_2/src/lightning.py`.

In [4]:
path = '~/seminar-project/project5'

In [5]:
!cat {path}/intro_mlops_2/src/lightning.py

# src/lightning.py

import pytorch_lightning as pl
import torch
from torch import nn

class WineQualityClassifier(pl.LightningModule):
    def __init__(self, model, learning_rate, epochs):
        super().__init__()
        self.save_hyperparameters(ignore=["model"]) # don't serialize the model object
        self.model = model
        self.loss_fn = nn.CrossEntropyLoss() # cross entropy loss is a common loss function for classification tasks

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch                      # splits out data (`x`) and labels (`y`)
        logits = self(x)                  # passes data into models forward method
        loss = self.loss_fn(logits, y)    # calcs loss based on logits (output of model) and ground truth
        self.log("train_loss", loss)      # this is a special method that logs the loss at each step
        return loss

    def configure_optimizers(self):
        # uses self

## Question 2

### Before deliverables

In [2]:
import pandas as pd

data = pd.read_csv('/anvil/projects/tdm/data/wine/wine_quality_type.csv')
print(data.head())

   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.56   

   alcohol  quality type  
0      9.4        5  red  
1      9.8        5  red  
2    

### 2.1 Run `!cat {path}/intro_mlops_2/src/data_loader.py | head -n 45`.

In [6]:
!cat {path}/intro_mlops_2/src/data_loader.py | head -n 45

# src/data_loader.py - project 5 version

import pytorch_lightning as pl
import pandas as pd
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

class WineQualityDataModule(pl.LightningDataModule):
    def __init__(self, data_path, batch_size=32, train_split=0.8): # you can add more parameters here if you want
        super().__init__()
        self.data_path = data_path
        self.batch_size = batch_size
        self.train_split = train_split

    def setup(self, stage=None):
        # Load and preprocess data - same logic from load_and_preprocess_data()
        data = pd.read_csv(self.data_path).dropna()

        # Create quality bins
        def bin_quality(quality):
            if quality <= 4:
                return 0  # Bad
            elif quality <= 7:
                return 1  # Mid
            else:
                return 2  # Good

  

### 2.2 Run `!cat {path}/intro_mlops_2/main.py`.

In [10]:
!cat {path}/intro_mlops_2/main.py

import pytorch_lightning as pl

from intro_mlops_2.src.lightning import WineQualityClassifier
from intro_mlops_2.src.data_loader import WineQualityDataModule
from intro_mlops_2.src.config import DATA_PATH, MODEL_PATH, PLOT_PATH, LOGS_PATH, \
    TRAIN_SPLIT, BATCH_SIZE, LEARNING_RATE, EPOCHS, INPUT_SIZE, NUM_CLASSES
from intro_mlops_2.src.neural_net import SimpleNN


def main():
    # Initialize model and dataloader
    simple_nn = SimpleNN(INPUT_SIZE, NUM_CLASSES)
    model = WineQualityClassifier(simple_nn, learning_rate=LEARNING_RATE, epochs=EPOCHS)
    datamodule = WineQualityDataModule(data_path=DATA_PATH / "wine_quality_type.csv", batch_size=BATCH_SIZE, train_split=TRAIN_SPLIT)

    # Initialize trainer object and train
    trainer = pl.Trainer(max_epochs=model.hparams.epochs)
    trainer.fit(model, datamodule=datamodule)

if __name__ == "__main__":
    main()


### 2.3 Run the pipline (in the notebook under the `intro_mlops_2/notebooks/` directory): `!cd ../../; python -m intro_mlops_2.main`.

In [11]:
!cd ../../; python -m intro_mlops_2.main

/usr/local/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
2026-02-25 18:26:17.341760: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772061977.356749 1201690 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772061977.361620 1201690 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1

## Question 3

In [3]:
# code here

Markdown notes and full English sentences and analysis written here.

## Question 4

In [4]:
# code here

Markdown notes and full English sentences and analysis written here.

## Question 5

In [5]:
# code here

Markdown notes and full English sentences and analysis written here.

## Pledge

By submitting this work I hereby pledge that this is my own, personal work. I've acknowledged in the designated place at the top of this file all sources that I used to complete said work, including but not limited to: online resources, books, and electronic communications. I've noted all collaboration with fellow students and/or TA's. I did not copy or plagiarize another's work.

> As a Boilermaker pursuing academic excellence, I pledge to be honest and true in all that I do. Accountable together – We are Purdue.

https://www.purdue.edu/odos/osrr/honor-pledge/
